In [ ]:
import os
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field
from typing import List, TypedDict
from langchain.schema import Document
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.prompts import ChatPromptTemplate
from pprint import pprint


In [ ]:
#Step 1:Load and prepare documents
urls = [
    "urls....."
]

#Load and split documents
docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=250, chunk_overlap=0)
doc_splits = text_splitter.split_documents(docs_list)
#Add to vectorstores
vectorstore = Chroma.from_documents(doc_splits, collection_name="rag-chroma", embedding=OpenAIEmbeddings())
retriever = vectorstore.as_retriever()

#Step 2: Define graders and relevance Model
class GradeDocuments(BaseModel):
    binary_score: str = Field(description="Documents are relevant to the question, 'yes' or 'no'")

retrieval_prompt = ChatPromptTemplate.from_template("""
You are a grader assessing if a document is relevant to a user's question.
Document: {document}
Question: {question}
Is the document relevant?Answer 'yes' or 'no'.
""")

retrieval_grader = retrieval_prompt | ChatOpenAI(model="gpt-4o-mini", temperature=0).with_structured_output(GradeDocuments)


#Step 3: Query Re-writer
class ImproveQuestion(BaseModel):
    improved_question: str = Field(description="Formulate an improved question.")

re_write_prompt =  ChatPromptTemplate.from_template("""
Here is the initial question:\n\n {question} \n Formulate an improved question.
""")   

query_rewriter = re_write_prompt | ChatOpenAI(model="gpt-4o-mini", temperature=0).with_structured_output(ImproveQuestion)


#Define prompt template
prompt = ChatPromptTemplate.from_template("""
Use the following context to answer the question:
Question: {question}
Context: {context}
Answer: 
""")

rag_chain = prompt | ChatOpenAI(model="gpt-4o-mini", temperature=0) | StrOutputParser()

#Define CRAG state
class GraphState(TypedDict):
    question: str
    generationn: str
    web_search: str
    documents: List[str]
    

#Step 4: Define workflow Nodes
def retrieve(state):
    """
    Retreive the documents
    Args:
        state(dict): The current graph state
    Returns:
        state(dict): New key added to state, documents, that contains retrieved documents
    """
    question = state["question"]

    #Retrieval
    documents = retriever.invoke(question)
    return {"documents": documents, "question": question}


def grade_documents(state):
    question = state["question"]
    documents = state["documents"]
    filtered_docs = []
    web_search_needed = 'No'
    for doc in documents:
        grade = retrieval_grader.invoke({"question": question, "document": doc.page_content}).binary_score
        if grade == "yes":
            print("---GRADE: DOCUMENT RELEVANT---")
            filtered_docs.append(doc)
        else:
            print("---GRADE: DOCUMENT NOT RELEVANT---")
            web_search_needed = 'Yes'
    return {"documents": filtered_docs, "question": question, "web_search": web_search_needed}


def transform_query(state):
        question = state["question"]
        rewritten_question = query_rewriter.invoke({"question":question})
        return {"question": rewritten_question.improved_question, "documents": state["documents"]}


def web_search(state):
    print("---web search---")
    question = state["question"]
    documents = state["documents"]
    pprint(question+"\n")
    search_results = TavilySearchResults(k=3).invoke({"query":question})
    #Process results to create document objects only with page_content
    web_documents = [
        Document(page_content=result["nontent"])for result in search_results if "nontent" in result
    ]
    #Append web search results to the existing documents
    documents.extend(web_documents)
    return {"documents": documents, "question": question}


def generate(state):
    
    #RAG Generation
    generation = rag_chain.invoke({"context": state["documents"], "question": state["question"]})
    return {"generation": generation}

#Step 5: Define decision-making logic
def decide_to_generate(state):
    """Determine weather to generate an answer or re generate a question."""
    print("---ASSESS GRADED DOCUMENTS---")
    state["question"]
    web_search = state["web_search"]
    state["documents"]
    if web_search == "Yes":
        return "transform_query"


#Step 6: Build and compile the Graph
workflow = StateGraph(GraphState)
workflow.add_node("retrieve", retrieve)
workflow.add_node("grade_documents", grade_documents)
workflow.add_node("transform_query", transform_query)
workflow.add_node("web_searcher", web_search)
workflow.add_node("generate",generate)
#Define edges
workflow.add_edge(START, "retrieve")
workflow.add_edge("retrieve", "grade_documents")
workflow.add_conditional_edges("grade_documents", decide_to_generate, {"transform_query": "transform_query", "generate": "generate"})
workflow.add_edge("transform_query", "web_searcher")
workflow.add_edge("web_searcher", "generate")
workflow.add_edge("generate", END)
app = workflow.compile()
